# RescueVision v2 - building segmentation (Unet++ EffB4, 608px)

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
import segmentation_models_pytorch as smp
print("smp", smp.__version__)
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

from rescuevision.constants import IMAGENET_MEAN, IMAGENET_STD
from rescuevision.data import datasets as _D
from PIL import Image as _PIL
from pathlib import Path as _PPath

def _load_arr(path):
    path = _PPath(path)
    if path.suffix == ".npy":
        return np.load(path)
    return np.array(_PIL.open(path))

def _find_tile(directory, tile_id):
    for ext in (".png", ".jpg", ".jpeg", ".npy"):
        p = _PPath(directory) / f"{tile_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{tile_id} in {directory}")

def _tile_dataset_init(self, processed_dir, split_csv, split="train", mask_type="building_binary",
                       transform=None, image_size=512, debug_limit=None):
    import pandas as pd
    self.processed_dir = _PPath(processed_dir)
    self.images_dir = self.processed_dir / "images" / "pre"
    self.pre_dir = self.processed_dir / "images" / "pre"
    self.post_dir = self.processed_dir / "images" / "post"
    if type(self).__name__ == "XBDDamageSegmentationDataset":
        self.masks_dir = self.processed_dir / "masks" / "damage_multiclass"
    elif type(self).__name__ == "XBDChangeDetectionDataset":
        self.masks_dir = self.processed_dir / "masks" / "change_binary"
    else:
        self.masks_dir = self.processed_dir / "masks" / mask_type
    self.transform = transform
    self.image_size = image_size
    splits_df = pd.read_csv(split_csv)
    tile_ids = splits_df[splits_df["split"] == split]["tile_id"].tolist()
    if debug_limit:
        tile_ids = tile_ids[:debug_limit]
    self.tile_ids = [t for t in tile_ids
                     if _find_tile(self.pre_dir, t).exists()
                     and _find_tile(self.post_dir, t).exists()
                     and _find_tile(self.masks_dir, t).exists()]
    print(f"{type(self).__name__}: split={split}, {len(self.tile_ids)} tiles")

def _xbd_seg_getitem(self, idx):
    tile_id = self.tile_ids[idx]
    image = _load_arr(_find_tile(self.pre_dir, tile_id))
    mask = _load_arr(_find_tile(self.masks_dir, tile_id))
    if self.transform:
        aug = self.transform(image=image, mask=mask)
        image, mask = aug["image"], aug["mask"]
    mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
    return {"image": image, "mask": mask, "tile_id": tile_id}

_D.XBDSegmentationDataset.__init__ = _tile_dataset_init
_D.XBDSegmentationDataset.__getitem__ = _xbd_seg_getitem
mark("patch:done")
print("png-aware dataset patch applied")

In [ ]:
import time
import pandas as pd
mark("tar:wait")
TAR = None
if (Path("/tmp/processed_xbd") / "metadata" / "splits_event_aware.csv").exists():
    print("processed tree already at /tmp/processed_xbd (smoke/debug path)")
else:
    for _attempt in range(40):
        for _p in Path("/kaggle/input").rglob("processed_xbd*.tar"):
            TAR = _p
            break
        if TAR is not None:
            break
        _ents = [p.name for p in Path("/kaggle/input").iterdir()] if Path("/kaggle/input").exists() else []
        print(f"[{_attempt+1}/40] waiting for tar mount, /kaggle/input: {_ents}", flush=True)
        time.sleep(30)
    assert TAR is not None, "processed_xbd_native.tar never mounted - kernel-source attach failed"
    _gib = TAR.stat().st_size / 1e9
    print("tar:", TAR, f"{_gib:.2f} GB")
    mark("tar:mounted", extra=f"{_gib:.2f}GB")
    subprocess.run(["tar", "-xf", str(TAR), "-C", "/tmp"], check=True)
PROCESSED = Path("/tmp/processed_xbd")
assert (PROCESSED / "metadata" / "splits_event_aware.csv").exists(), "extracted tree missing splits csv"
_splits = pd.read_csv(PROCESSED / "metadata" / "splits_event_aware.csv")
print("tiles per split:", _splits["split"].value_counts().to_dict())
_tiles_meta = pd.read_csv(PROCESSED / "metadata" / "tiles.csv")
print("shift columns:", [c for c in _tiles_meta.columns if "shift" in c])
mark("tar:extracted", extra=str(_splits["split"].value_counts().to_dict()))

In [ ]:
import logging
import pandas as pd
from torch.utils.data import DataLoader
from rescuevision.data.datasets import XBDSegmentationDataset
from rescuevision.models.losses import BCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation, evaluate_segmentation
from rescuevision.utils.seed import set_seed

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
name = "building_seg_unetpp_effb4"
SIZE = 608
EPOCHS = 30
PATIENCE = 7
BS = 8
ACCUM = 1

def bld_transforms(phase="train", size=608):
    tf = [A.Resize(size, size)]
    if phase == "train":
        tf += [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
               A.Affine(scale=(0.85, 1.15), p=0.4), A.RandomBrightnessContrast(p=0.3)]
    tf += [A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return A.Compose(tf, is_check_shapes=False)

set_seed(42)
split_csv = PROCESSED / "metadata" / "splits_event_aware.csv"
train_ds = XBDSegmentationDataset(PROCESSED, split_csv, split="train", mask_type="building_binary",
                                  transform=bld_transforms("train", SIZE))
val_ds = XBDSegmentationDataset(PROCESSED, split_csv, split="val", mask_type="building_binary",
                                transform=bld_transforms("val", SIZE))
test_ds = XBDSegmentationDataset(PROCESSED, split_csv, split="test", mask_type="building_binary",
                                 transform=bld_transforms("val", SIZE))
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
mark("bldv2:data", extra=f"train={len(train_ds)}")

model = getattr(smp, "UnetPlusPlus")(encoder_name="efficientnet-b4", encoder_weights=None,
                               in_channels=3, classes=1).to(DEVICE)
loss_fn = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)

config = {"seed": 42, "device": DEVICE.type,
          "model": {"arch": "UnetPlusPlus", "encoder": "efficientnet-b4"},
          "training": {"epochs": EPOCHS, "lr": 2e-4, "weight_decay": 1e-4, "scheduler": "cosine",
                       "mixed_precision": True, "early_stopping_patience": PATIENCE,
                       "save_best_metric": "iou", "gradient_accumulation_steps": ACCUM}}

def _make_loaders(bs):
    return (DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW)),
            DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NW, pin_memory=False, persistent_workers=bool(NW)),
            DataLoader(test_ds, batch_size=bs, shuffle=False, num_workers=NW, pin_memory=False))

# probe with OOM fallback (Unet++ EffB4 @608 is the heaviest thing we train)
for _attempt in range(3):
    train_loader, val_loader, test_loader = _make_loaders(BS)
    config["training"]["gradient_accumulation_steps"] = ACCUM
    try:
        _it = iter(train_loader)
        for _ in range(2):
            _b = next(_it)
            _img = _b["image"].to(DEVICE); _m = _b["mask"].to(DEVICE)
            with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
                _lg = model(_img)
                _loss = loss_fn(_lg.squeeze(1), _m.float())
            _loss.backward()
            model.zero_grad(set_to_none=True)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
            mark("probe:ok", extra=f"bs={BS} peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
        else:
            mark("probe:ok", extra=f"cpu bs={BS}")
        break
    except RuntimeError as _e:
        if "out of memory" in str(_e).lower() and DEVICE.type == "cuda" and BS > 2:
            torch.cuda.empty_cache()
            BS //= 2; ACCUM *= 2
            mark("probe:oom-retry", extra=f"bs->{BS} accum->{ACCUM}")
            continue
        mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
        raise

mark("bldv2:train-start", extra=f"train={len(train_ds)} bs={BS}")
best = train_segmentation(model, train_loader, val_loader, loss_fn, config,
                          output_dir=str(Path("/tmp/run") / name), is_binary=True, num_classes=1)
print("best:", best)
mark("bldv2:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/building_seg_unetpp_effb4", WORK / "building_seg_unetpp_effb4", dirs_exist_ok=True)
print("results copied to", WORK / "building_seg_unetpp_effb4")
mark("copyout:building_seg_unetpp_effb4")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "building_seg_unetpp_effb4" / "logs").glob("*.csv")))
metric = "iou"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
if metric in log_df.columns:
    axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
    axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ ep {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "building_seg_unetpp_effb4" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json
# ---- shared eval kit: threshold sweep + approximate official xView2 score ----
import numpy as np
from scipy.ndimage import label as _cc_label

THRS = np.arange(0.20, 0.81, 0.05)

def sweep_loc_thresholds(probs, gts):
    """pooled inter/union per threshold -> {thr: IoU}"""
    out = {}
    p = np.concatenate(probs); g = np.concatenate(gts).astype(bool)
    for t in THRS:
        pred = p > t
        inter = np.logical_and(pred, g).sum()
        union = np.logical_or(pred, g).sum()
        out[round(float(t), 2)] = float(inter) / max(union, 1)
    return out

def best_thr(probs, gts):
    pooled = sweep_loc_thresholds(probs, gts)
    thr = max(pooled, key=pooled.get)
    return float(thr), pooled[thr], pooled

def _xview2_tile(loc_pred, loc_gt, dmg_pred, dmg_gt):
    """greedy component match: returns (fp, fn, matches=[(iou, pred_grade, gt_grade)])"""
    p_lab, np_ = _cc_label(loc_pred)
    g_lab, ng_ = _cc_label(loc_gt)
    fp = 0
    matches = []
    used = np.zeros(ng_ + 1, dtype=bool)
    if np_ and ng_:
        pair = p_lab.astype(np.int64) * (ng_ + 1) + g_lab
        hist = np.bincount(pair.ravel(), minlength=(np_ + 1) * (ng_ + 1)).reshape(np_ + 1, ng_ + 1)
        areas_p = hist[1:].sum(axis=1); areas_g = hist[:, 1:].sum(axis=0)
        for pi in range(np_):
            inters = hist[pi + 1, 1:]
            gi = int(np.argmax(inters))
            inter = int(inters[gi])
            if inter == 0:
                fp += 1
                continue
            union = int(areas_p[pi] + areas_g[gi] - inter)
            iou = inter / union
            if iou >= 0.5 and not used[gi]:
                used[gi] = True
                m = hist[pi + 1, gi + 1] > 0
                pg = dmg_pred[m]; gg = dmg_gt[m]
                pg = pg[pg > 0]; gg = gg[gg > 0]
                pgr = int(np.bincount(pg).argmax()) if len(pg) else 0
                ggr = int(np.bincount(gg).argmax()) if len(gg) else 0
                matches.append((iou, pgr, ggr))
            else:
                fp += 1
    else:
        fp += int(np_)
    fn = int((~used[1:]).sum())
    return fp, fn, matches

def xview2_score(tiles, thr):
    """tiles: [{loc_prob,loc_gt,dmg_pred,dmg_gt}] -> approximate official xView2 score."""
    stats = {t: [0, 0, 0] for t in (0.5, 0.6, 0.7, 0.8)}   # tp, fp, fn
    matches50 = []
    for d in tiles:
        pred = d["loc_prob"] > thr
        fp, fn, matches = _xview2_tile(pred, d["loc_gt"], d["dmg_pred"], d["dmg_gt"])
        for t in stats:
            tp_t = sum(1 for iou, _, _ in matches if iou >= t)
            fp_t = fp + sum(1 for iou, _, _ in matches if iou < t)
            fn_t = fn + sum(1 for iou, _, _ in matches if iou < t)
            stats[t][0] += tp_t; stats[t][1] += fp_t; stats[t][2] += fn_t
        matches50 += matches
    _f1 = lambda tp, fp, fn: 2 * tp / max(2 * tp + fp + fn, 1)
    loc_f1 = float(np.mean([_f1(*stats[t]) for t in stats]))
    per_class = {}
    for c in (1, 2, 3, 4):
        tp_c = sum(1 for _, p, g in matches50 if p == c and g == c)
        fp_c = sum(1 for _, p, g in matches50 if p == c and g != c)
        fn_c = sum(1 for _, p, g in matches50 if p != c and g == c)
        per_class[c] = round(2 * tp_c / max(2 * tp_c + fp_c + fn_c, 1), 4)
    dmg_f1 = float(np.mean(list(per_class.values())))
    return {"loc_f1": round(loc_f1, 4), "damage_f1": round(dmg_f1, 4),
            "overall_f1": round(0.3 * loc_f1 + 0.7 * dmg_f1, 4),
            "per_class_damage_f1": per_class,
            "n_pred_buildings": int(sum(s[0] + s[1] for s in stats.values()) // 8)}
print("eval kit ready")

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print("best ckpt: epoch", ckpt["epoch"], "iou", ckpt.get("best_iou"))

val_metrics = evaluate_segmentation(model, val_loader, DEVICE, is_binary=True, num_classes=1)
test_metrics = evaluate_segmentation(model, test_loader, DEVICE, is_binary=True, num_classes=1)
print("VAL @0.5 :", {k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)})
print("TEST @0.5:", {k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)})

@torch.no_grad()
def bld_probs(ds, limit=None):
    if limit:
        ds.tile_ids = ds.tile_ids[:limit]
    loader = DataLoader(ds, batch_size=BS, shuffle=False, num_workers=NW)
    probs, gts = [], []
    for b in loader:
        img = b["image"].to(DEVICE)
        outs = []
        for k in range(4):
            x = img
            if k == 1: x = torch.flip(img, [3])
            elif k == 2: x = torch.flip(img, [2])
            elif k == 3: x = torch.flip(img, [2, 3])
            with torch.amp.autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
                o = model(x)
            if k == 1: o = torch.flip(o, [3])
            elif k == 2: o = torch.flip(o, [2])
            elif k == 3: o = torch.flip(o, [2, 3])
            outs.append(o)
        p = torch.sigmoid(torch.stack(outs, 0).mean(0).float().squeeze(1)).cpu().numpy()
        probs.append(p); gts.append(b["mask"].numpy() > 0.5)
    return probs, gts

p, g = bld_probs(val_ds, limit=250)
thr, thr_iou, pooled = best_thr(p, g)
print(f"thr sweep: best={thr:.2f} val IoU={thr_iou:.4f} (0.5 gives {pooled.get(0.5)})")

p2, g2 = bld_probs(test_ds)
pi = sweep_loc_thresholds(p2, g2)
test_iou_tta = pi[round(thr, 2)]
print(f"TEST TTA IoU @{thr}: {test_iou_tta:.4f} | @0.5: {pi.get(0.5)}")

# approx official building-localization F1 on up to 300 test tiles
sub = p2[:300]; subg = g2[:300]
tp = fp = fn = 0
for i in range(len(sub)):
    _fp, _fn, matches = _xview2_tile(sub[i] > thr, subg[i], np.zeros_like(subg[i], dtype=np.int8),
                                     np.zeros_like(subg[i], dtype=np.int8))
    tp += len(matches); fp += _fp; fn += _fn
loc_f1 = 2 * tp / max(2 * tp + fp + fn, 1)
print(f"approx official loc F1 @0.5 IoU: {loc_f1:.4f} ({tp} TP / {fp} FP / {fn} FN)")

summary = {"run": name, "best_epoch": int(ckpt["epoch"]),
           "best_val_iou": float(ckpt.get("best_iou")),
           "val_at_0.5": {k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)},
           "test_at_0.5": {k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)},
           "threshold": thr, "test_tta_iou_at_thr": round(float(test_iou_tta), 4),
           "test_tta_iou_at_0.5": round(float(pi.get(0.5, 0)), 4),
           "approx_loc_f1": round(float(loc_f1), 4)}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import numpy as np, matplotlib.pyplot as plt
batch = next(iter(DataLoader(test_ds, batch_size=4, shuffle=True, num_workers=NW)))
with torch.no_grad():
    pred = (torch.sigmoid(model(batch["image"].to(DEVICE))) > thr).cpu().numpy()
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i in range(min(4, batch["image"].shape[0])):
    img = batch["image"][i].numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    axes[0, i].imshow(img); axes[0, i].set_title(batch["tile_id"][i][:28])
    axes[1, i].imshow(batch["mask"][i], cmap="gray"); axes[1, i].set_title("GT")
    axes[2, i].imshow(pred[i, 0], cmap="gray"); axes[2, i].set_title("pred")
    for ax in axes[:, i]: ax.axis("off")
plt.tight_layout(); plt.savefig(WORK / name / "test_predictions.png", dpi=120, bbox_inches="tight")
plt.show()
mark("bldv2:eval-done")